# Day 89 — Debugging

This notebook covers Neural Network Debugging & Sanity Checks, Single-Batch Overfit Verification (`run_single_batch_overfit_test`), Tensor Health Diagnostics (`check_tensor_health`), Global Gradient L2 Norm Monitoring (`compute_total_grad_norm`), and Silent Bug Prevention

## Validation goals
- Implement tensor health checks (`check_tensor_health`) to verify shapes, dtypes, min/max ranges, and absence of `NaN` or `Inf` values before training
- Build a global L2 gradient norm diagnostic (`compute_total_grad_norm`) to detect vanishing or exploding gradients across model parameters
- Execute a single-batch memorization sanity check (`run_single_batch_overfit_test`) with `dropout=0.0` to isolate model, loss, and optimizer wiring from full-dataset noise
- Diagnose common neural network failure modes including `NaN` loss divergence, unintended loss broadcasting, un-zeroed gradient accumulation, and stochastic dropout variance during sanity tests

### 1. High-Signal Debugging Checks for Neural Networks

When a training run behaves unexpectedly, run targeted checks in order rather than guessing hyperparameters at random:

- Overfit a Single Mini-Batch First: Take one small batch (for example, 8–16 samples) and train for many iterations. If a network cannot drive loss near zero on a tiny batch, check model capacity, learning rate, target scaling, loss wiring, or label alignment before touching the full dataset.
- Audit Tensor Shapes and Dtypes: Confirm `preds` and `targets` have matching shapes (`(batch_size, 1)`) and compatible dtypes (`torch.float32`), especially on the final partial batch when drop_last=False.
- Check for `NaN` / `Inf` and Exploding Gradients: Inspect inputs, targets, losses, and parameter gradients (param.grad). Unscaled features, high learning rates, or division/log edge cases often trigger `NaNs`.
- Watch Learning Rate & Convergence Behavior:
    - Loss explodes or turns NaN -> learning rate is likely too high (or inputs/targets have huge scale/outliers).
    - Loss barely moves -> learning rate may be too low, gradients may be vanishing/zeroed, or optimizer.zero_grad() / loss.backward() / optimizer.step() order is broken.
    - Loss oscillates wildly -> learning rate or batch noise is high; try a smaller learning rate or larger batch size.
- Verify Train vs. Eval Mode & Device Consistency: Ensure all tensors and the model share device, and check that model.eval() + torch.no_grad() is used during evaluation.

### 2. Diagnostic Helpers & Single-Batch Overfit Test (`day_89_debugging.py`)

Let's write a practical diagnostic module with a single-batch overfit test, a tensor/gradient health audit, and gradient norm inspection so you have concrete tools to drop into any PyTorch script.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from day_85_dataset import build_dataloaders
from day_86_architecture import RetailDemandMLP
from day_87_training import set_seed


def check_tensor_health(tensor: torch.Tensor, name: str) -> bool:
    """Checks a tensor for NaN or Inf values and prints shape/range diagnostics."""
    has_nan = bool(torch.isnan(tensor).any().item())
    has_inf = bool(torch.isinf(tensor).any().item())
    ok = not (has_nan or has_inf)

    t_min = tensor.min().item() if tensor.numel() > 0 else float("nan")
    t_max = tensor.max().item() if tensor.numel() > 0 else float("nan")

    print(
        f"[Health] {name:18s} | shape={str(tuple(tensor.shape)):12s} | "
        f"dtype={str(tensor.dtype):14s} | min={t_min:8.2f} | max={t_max:8.2f} | "
        f"nan={has_nan} | inf={has_inf}"
    )
    return ok


def compute_total_grad_norm(model: nn.Module) -> float:
    """Computes the global L2 norm across all parameter gradients."""
    sq_sum = 0.0
    for param in model.parameters():
        if param.grad is not None:
            grad_norm = param.grad.detach().data.norm(2).item()
            sq_sum += grad_norm ** 2
    return float(np.sqrt(sq_sum))


def run_single_batch_overfit_test(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    steps: int = 200,
    lr: float = 1e-2,
) -> dict[str, float]:
    """Verifies that the model and loss can memorize a single mini-batch."""
    model.train()
    batch_x, batch_y = next(iter(dataloader))
    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)

    check_tensor_health(batch_x, "single_batch_x")
    check_tensor_health(batch_y, "single_batch_y")

    # Fresh optimizer isolated to this sanity check
    probe_optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    initial_loss = None
    final_loss = None
    final_grad_norm = 0.0

    for step in range(1, steps + 1):
        probe_optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)

        if torch.isnan(loss) or torch.isinf(loss):
            raise RuntimeError(f"Non-finite loss detected at step {step}: {loss.item()}")

        loss.backward()
        final_grad_norm = compute_total_grad_norm(model)
        probe_optimizer.step()

        loss_val = float(loss.item())
        if initial_loss is None:
            initial_loss = loss_val
        final_loss = loss_val

    return {
        "initial_mse": initial_loss if initial_loss is not None else float("nan"),
        "final_mse": final_loss if final_loss is not None else float("nan"),
        "final_rmse": float(np.sqrt(final_loss)) if final_loss is not None else float("nan"),
        "final_grad_norm": final_grad_norm,
    }


if __name__ == "__main__":
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, _, num_features, source = build_dataloaders(batch_size=16)
    model = RetailDemandMLP(
        in_features=num_features,
        hidden_dims=(64, 32),
        dropout=0.0,  # Keep dropout off during a single-batch memorization check
    ).to(device)

    criterion = nn.MSELoss()

    print("--- Day 89: Neural Network Debugging & Sanity Checks ---")
    print(f"Data source: {source} | Device: {device} | Features: {num_features}\n")

    stats = run_single_batch_overfit_test(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        device=device,
        steps=250,
        lr=2e-2,
    )

    print("\n--- Single-Batch Overfit Summary ---")
    print(f"Initial MSE:      {stats['initial_mse']:10.2f}")
    print(f"Final MSE:        {stats['final_mse']:10.4f}")
    print(f"Final RMSE:       {stats['final_rmse']:8.4f} units")
    print(f"Final Grad Norm:  {stats['final_grad_norm']:8.4f}")

--- Day 89: Neural Network Debugging & Sanity Checks ---
Data source: synthetic_fallback | Device: cpu | Features: 12

[Health] single_batch_x     | shape=(16, 12)     | dtype=torch.float32  | min=   -2.57 | max=    2.21 | nan=False | inf=False
[Health] single_batch_y     | shape=(16, 1)      | dtype=torch.float32  | min=   74.63 | max=  222.02 | nan=False | inf=False

--- Single-Batch Overfit Summary ---
Initial MSE:        24889.70
Final MSE:            0.2597
Final RMSE:         0.5096 units
Final Grad Norm:    2.8219


### 3. Common Silent Bugs to Catch Early

- Unintended Broadcasting in Loss: If preds has shape (`32, 1`) and targets has shape (`32,`) in certain expressions, element-wise operations can expand to (`32, 32`) instead of (`32, 1`). Always assert or verify shapes during debugging.
- Forgetting `optimizer.zero_grad()`: Causes gradients to accumulate across batches, leading to runaway updates or distorted step directions.
- Evaluating Without `model.eval()`: Leaves Dropout active during validation/test passes, inflating error slightly and adding randomness to evaluation runs.
- Data Leakage or Unscaled Skewed Inputs: Extreme feature values or unaligned target columns often show up immediately as unstable early loss or `NaN` gradients.

### Guided Question:

Why is overfitting a single mini-batch first one of the fastest ways to isolate model/optimizer/loss bugs from dataset-wide issues?

### Answer:

Overfitting a single mini-batch (typically 1 to 32 examples) until training loss approaches zero acts as a sanity check on your training loop and model capacity before you spend time debugging data pipelines, train/test splits, or large-scale convergence.

**Why It Isolates Bugs So Quickly**

- Eliminates data-scale and distribution noise: Across a full dataset, slow or noisy loss can come from class imbalance, label noise, mislabeled samples, broken augmentations, or batch-to-batch variance. On a tiny fixed batch, the model sees the exact same inputs repeatedly, so data-wide quirks cannot hide behind gradient averaging.
- Tests raw expressiveness and gradient flow: A modern neural network has plenty of capacity to memorize a small batch. If loss refuses to drop, you immediately suspect broken gradient flow (vanishing/exploding gradients, detached tensors, wrong activation/loss pairing, or frozen parameters) rather than "the model needs more data."
- Catches loss, target, and metric mismatches early: Shape bugs (such as (`B,`) vs. `(B, 1)`), unshifted logits passed to a loss expecting probabilities (or vice versa), wrong label indexing, or swapped target/prediction arguments show up fast when loss plateaus or behaves strangely on a known sample.
- Checks optimizer step and learning rate scale: If the learning rate is far too high, loss explodes or oscillates wildly even on one batch; if it is far too low or the optimizer isn't updating weights properly, loss barely moves.
- Gives rapid feedback cycles: One batch runs in milliseconds to a couple of seconds per epoch, letting you test fixes quickly instead of waiting through full epochs.

**Practical Caveats**
- Memorization is not generalization: Overfitting one batch proves your optimization mechanics can fit data; it does not prove your features generalize or your full dataset is clean.
- BatchNorm behavior: With very small batch sizes (especially batch size 1 during training or eval mode mismatches), batch normalization statistics can behave oddly, so test with a small multi-sample batch (e.g., 8–32 items) when possible.

### Guided Question:

If loss becomes `NaN` after a few steps, what are two likely causes in the data pipeline or optimizer setup, and how would you check them?

### Answer:

When training loss turns into `NaN` after a few steps, the issue usually comes down to numerical instability turning an intermediate value into inf or `NaN`, which then propagates through gradients and weights.

**1. Data Pipeline: Unnormalized, Extreme, or Corrupted Inputs/Targets**

Large input features, extreme outliers, or invalid target values can push activations or logits into ranges where exponentiation, division, or logarithms overflow or become undefined. For example, large unscaled features feeding into `exp()`, `log()`, or division/normalization layers can quickly produce `inf`, and `inf - inf` or `0 / 0` yields `NaN`. Mislabeled or out-of-range targets (such as negative class indices or targets outside [0, 1] for binary cross-entropy) can also trigger `NaN` inside loss functions.

**How to check it:**

- Inspect summary statistics on a batch before the forward pass: check `min`, `max`, `mean`, standard deviation, and count of `NaN` or `inf` values in both inputs x and targets y.
- Verify target value ranges and data types against what the loss function expects (for example, valid integer class indices or valid probabilities/logits).
- Temporarily log or assert `torch.isfinite(x).all()` and `torch.isfinite(y).all()` inside the training loop to catch bad batches immediately.

**2. Optimizer Setup: Learning Rate Too High (or Unscaled/Unclipped Large Gradients)**

If the learning rate is too high—especially early in training without warmup, or with adaptive/momentum updates—a single steep gradient step can overshoot and push weights or activations to huge values. Once activations or loss terms hit `inf`, `subsequent` backward passes multiply by `inf` or divide `inf / inf`, locking weights and gradients into `NaN`.

**How to check it:**

- Lower the learning rate by an order of magnitude (for example, from `1e-3 to 1e-4` or `3e-4`) and see if the NaN disappears or takes longer to occur.
- Check gradient norms per step before `optimizer.step()`; a sudden spike in gradient norm right before loss becomes NaN points to an overly aggressive step size or unstable gradient path.
- Test temporarily enabling gradient norm clipping as a diagnostic probe to see whether capping extreme updates stabilizes the first few steps while you tune the learning rate or model scaling.

### Guided Question:

Why is setting `dropout=0.0` (or using `model.eval()`) helpful when testing whether a small network can fit a single batch cleanly?

### Answer:

Setting dropout=0.0 (or calling `model.eval()`) removes stochastic noise and training-mode shifts so you can test pure memorization cleanly without regularizers fighting against fitting a single batch.

**Why It Helps During a Single-Batch Sanity Check**

- Removes random neuron zeroing: Dropout randomly drops activations during training, which acts as a regularizer and adds variance to each forward pass. When testing if a network has enough capacity to overfit a small batch, random masking slows convergence or makes loss bounce around even when the model and optimizer are healthy.
- Eliminates `train-vs-eval` mode discrepancies: Calling `model.eval()` switches off layers like Dropout and makes `BatchNorm` use running statistics rather than batch statistics (when track_running_stats=True). Note the trade-off: `model.eval()` affects all mode-dependent layers at once, whereas setting `dropout=0.0` targets dropout specifically while leaving `BatchNorm` in training mode if you want batch statistics on your test batch.
- Gives cleaner, more deterministic loss curves: With dropout active, loss can fluctuate from step to step even on identical inputs. Disabling dropout makes loss decrease smoothly if your learning rate, loss function, and gradients are working properly.

**Practical Caveat**

Restore training settings afterward: Overfitting a single batch with `dropout=0.0` or `model.eval()` only tests whether the architecture and optimization loop can memorize data. Once that sanity check passes, restore your intended dropout rate and `model.train()` before training on the full dataset so regularization works as designed.